## Air Quality Index (AQI) — 3D Terrain Visualization with rayshader

This notebook queries AQI data for the Pyrenees corridor — the French regions of Occitanie and Nouvelle-Aquitaine, and the Spanish regions of Aragon, Catalonia and Navarre — from Neo4j and renders a path-traced 3D terrain map using `rayshader`. City AQI spikes and the France-Spain international border are overlaid on the rendered terrain.

Terrain elevation is fetched from AWS Open Data Terrain Tiles via `elevatr`. See the article appendix for installation notes and gotchas on Apple Silicon macOS.

### Before running the notebook

Export the following in your shell:

```bash
export NEO4J_URI=bolt://127.0.0.1:7687
export NEO4J_USERNAME=your_username_here
export NEO4J_PASSWORD=your_password_here
export NEO4J_DATABASE=your_database_name_here
```

### Install and Load Packages

In [1]:
# Install spatial and mapping packages from CRAN.
# rayshader is installed from r-universe.
#
# Note: this notebook uses render_snapshot() only — it does not require
# the rayrender/libopenexr/libimath stack. Those are only needed if you
# want to use render_highquality() for path-traced output.

suppressMessages(install.packages(
  c("elevatr", "httr2", "rnaturalearth", "rnaturalearthdata", "sf", "terra"),
  quiet = TRUE
))

suppressMessages(install.packages(
  "rayshader",
  repos = "https://tylermorganwall.r-universe.dev",
  quiet = TRUE
))

cat("Install complete.\n")

Install complete.


In [2]:
suppressMessages({
  library(elevatr)
  library(httr2)
  library(rayshader)
  library(rnaturalearth)
  library(rnaturalearthdata)
  library(sf)
  library(terra)
})

In [3]:
host     <- gsub("bolt://", "http://", Sys.getenv("NEO4J_URI"), fixed = TRUE)
host     <- gsub(":7687", ":7474", host, fixed = TRUE)
user     <- Sys.getenv("NEO4J_USERNAME")
password <- Sys.getenv("NEO4J_PASSWORD")
db       <- Sys.getenv("NEO4J_DATABASE")

cat("Credentials set.\n")

Credentials set.


### Query Helper Function

In [4]:
neo4j_query <- function(cypher) {
  response <- request(paste0(host, "/db/", db, "/query/v2")) |>
    req_auth_basic(user, password) |>
    req_headers(
      "Content-Type" = "application/json",
      "Accept"       = "application/json"
    ) |>
    req_body_json(list("statement" = cypher)) |>
    req_perform()

  json   <- resp_body_json(response, simplifyVector = TRUE)
  fields <- json$data$fields
  vals   <- json$data$values

  # Handle both matrix (simplifyVector worked) and list-of-lists
  if (is.matrix(vals)) {
    values <- as.data.frame(vals)
  } else {
    values <- as.data.frame(do.call(rbind, lapply(vals, function(row) {
      as.data.frame(t(unlist(row)), stringsAsFactors = FALSE)
    })))
  }

  colnames(values) <- fields
  values <- type.convert(values, as.is = TRUE)
  values
}

### Step 1: Query Latest AQI per City from Neo4j

We fetch the most recent Reading for each City node, along with its coordinates and AQI category.

In [5]:
aqi_data <- neo4j_query("
  MATCH (c:City)-[:HAS_READING]->(r:Reading)
  WITH c, r ORDER BY r.timestamp DESC
  WITH c, collect(r)[0] AS latest
  RETURN c.name    AS city,
         c.country AS country,
         c.lat     AS lat,
         c.lon     AS lon,
         latest.aqi_us       AS aqi_us,
         latest.aqi_category AS aqi_category
  ORDER BY latest.aqi_us DESC
")

cat(sprintf("Cities loaded: %d\n", nrow(aqi_data)))
cat(sprintf("AQI range: %d - %d\n", min(aqi_data$aqi_us), max(aqi_data$aqi_us)))

Cities loaded: 102
AQI range: 1 - 157


### Step 2: Define AQI Color Scale

We use the standard US AQI color scale so the visualization is immediately readable to anyone familiar with air quality reporting.

In [6]:
aqi_color <- function(aqi) {
  ifelse(aqi <= 50,  "#00E400",
  ifelse(aqi <= 100, "#FFFF00",
  ifelse(aqi <= 150, "#FF7E00",
  ifelse(aqi <= 200, "#FF0000",
  ifelse(aqi <= 300, "#8F3F97",
                     "#7E0023")))))
}

aqi_data$color <- aqi_color(aqi_data$aqi_us)

# Print the legend
cat("AQI color scale:\n")
cat("  0-50   Good                #00E400\n")
cat("  51-100 Moderate            #FFFF00\n")
cat(" 101-150 Unhealthy Sensitive #FF7E00\n")
cat(" 151-200 Unhealthy           #FF0000\n")
cat(" 201-300 Very Unhealthy      #8F3F97\n")
cat(" 301+    Hazardous           #7E0023\n")

AQI color scale:
  0-50   Good                #00E400
  51-100 Moderate            #FFFF00
 101-150 Unhealthy Sensitive #FF7E00
 151-200 Unhealthy           #FF0000
 201-300 Very Unhealthy      #8F3F97
 301+    Hazardous           #7E0023


### Step 3: Fetch Terrain Elevation

We define a bounding box covering the Pyrenees corridor across southern France and northern Spain, then fetch elevation tiles from AWS Open Data at zoom level 7 (roughly 1 km resolution, sufficient for a regional map of this scale).

In [7]:
# Bounding box: Pyrenees corridor covering southern France and northern Spain
bbox_sf <- st_as_sf(st_as_sfc(st_bbox(
  c(xmin = -2.0, ymin = 41.0, xmax = 5.5, ymax = 46.0),
  crs = st_crs(4326)
)))

# Fetch elevation tiles from AWS Open Data
# z = 7 gives ~1 km resolution; suppressMessages keeps elevatr progress quiet
elev <- suppressMessages(
  get_elev_raster(bbox_sf, z = 7, clip = "bbox")
)

elev_matrix <- raster_to_matrix(elev)
cat(sprintf("Elevation matrix: %d rows x %d cols\n", nrow(elev_matrix), ncol(elev_matrix)))
cat(sprintf("Elevation range:  %.0f - %.0f m\n", min(elev_matrix, na.rm = TRUE), max(elev_matrix, na.rm = TRUE)))

Elevation matrix: 1572 rows x 1050 cols
Elevation range:  -2698 - 3253 m


### Step 4: Build the Terrain Texture

We apply sphere shading and ambient occlusion to produce a natural-looking hillshade. The imhof1 texture suits the varied terrain of the Pyrenees corridor.

In [8]:
# Snow effect — must go last, after shadows
snow_hs <- height_shade(elev_matrix, texture = "white")

# Contour overlay for topographic feel
contour_overlay <- generate_contour_overlay(
  elev_matrix,
  color     = "grey30",
  linewidth = 0.5,
  levels    = seq(200, 3200, by = 200)
)

texture <- elev_matrix |>
  sphere_shade(texture = "imhof1") |>
  add_shadow(ray_shade(elev_matrix, zscale = 30), 0.5) |>
  add_shadow(ambient_shade(elev_matrix), 0.4) |>
  add_overlay(contour_overlay, alphalayer = 0.3) |>
  add_overlay(generate_altitude_overlay(snow_hs, elev_matrix,
              start_transition = 1800,
              end_transition   = 2400,
              lower            = FALSE), alphalayer = 0.95)

cat("Terrain texture built.\n")

Terrain texture built.


### Step 5: Convert City Coordinates to Matrix Space

rayshader works in matrix row/column space rather than geographic coordinates. We convert each city's lat/lon to the corresponding position in the elevation matrix.

In [9]:
ext   <- terra::ext(elev)
nrows <- nrow(elev_matrix)
ncols <- ncol(elev_matrix)

aqi_data$col_pos <- round((aqi_data$lon - ext[1]) / (ext[2] - ext[1]) * ncols)
aqi_data$row_pos <- round((ext[4] - aqi_data$lat) / (ext[4] - ext[3]) * nrows)

# Clamp to matrix bounds
aqi_data$col_pos <- pmax(1, pmin(ncols, aqi_data$col_pos))
aqi_data$row_pos <- pmax(1, pmin(nrows, aqi_data$row_pos))

cat(sprintf("City positions mapped: %d\n", nrow(aqi_data)))

City positions mapped: 102


In [10]:
# Filter to cities within the bounding box
aqi_data <- aqi_data[
  aqi_data$lon >= (ext[1] + 0.2) & aqi_data$lon <= (ext[2] - 0.2) &
  aqi_data$lat >= (ext[3] + 0.2) & aqi_data$lat <= (ext[4] - 0.2),
]

cat(sprintf("Cities within tile: %d\n", nrow(aqi_data)))

Cities within tile: 85


### Step 6: Render the 3D Scene

We render the terrain in 3D, then add a vertical spike at each city. Each spike is represented as a two-point vertical path: the base anchors at terrain elevation and the top rises proportionally to AQI value. Color follows the standard AQI scale. The camera is positioned to show both the Pyrenees ridge and the surrounding plains.

In [11]:
zscale          <- 40
spike_scale     <- 120
plains_max_elev <- 1000

plot_3d(
  texture,
  elev_matrix,
  zscale         = zscale,
  fov            = 45,
  theta          = 45,
  phi            = 35,
  zoom           = 0.38,
  windowsize     = c(2400, 1600),
  background     = "#ffffff",
  shadowcolor    = "#cccccc",
  water          = TRUE,
  waterdepth     = 0,
  watercolor     = "lightblue",
  wateralpha     = 0.8,
  waterlinecolor = "white",
  waterlinealpha = 0.5
)

# AQI spikes — each spike is a two-point vertical path:
# the first point anchors at terrain elevation, the second rises by AQI * spike_scale
for (i in seq_len(nrow(aqi_data))) {
  base_elev <- min(elev_matrix[aqi_data$row_pos[i], aqi_data$col_pos[i]], plains_max_elev)
  render_path(
    extent    = ext,
    lat       = c(aqi_data$lat[i], aqi_data$lat[i]),
    long      = c(aqi_data$lon[i], aqi_data$lon[i]),
    altitude  = c(base_elev, base_elev + aqi_data$aqi_us[i] * spike_scale),
    zscale    = zscale,
    color     = aqi_data$color[i],
    linewidth = 12,
    heightmap = elev_matrix
  )
}

# France-Spain border — terrain-following red line
border_lines <- ne_download(
  scale       = 50,
  type        = "boundary_lines_land",
  category    = "cultural",
  returnclass = "sf"
)

region_bbox <- st_bbox(
  c(xmin = -2.0, ymin = 41.0, xmax = 5.5, ymax = 44.5),
  crs = st_crs(4326)
)

border_cropped <- suppressWarnings(st_crop(border_lines, region_bbox))

for (i in seq_len(nrow(border_cropped))) {
  seg <- st_coordinates(border_cropped[i, ])
  if (nrow(seg) >= 2) {
    col_pos <- round((seg[, 1] - ext[1]) / (ext[2] - ext[1]) * ncols)
    row_pos <- round((ext[4] - seg[, 2]) / (ext[4] - ext[3]) * nrows)
    col_pos <- pmax(1, pmin(ncols, col_pos))
    row_pos <- pmax(1, pmin(nrows, row_pos))
    terrain_elev <- mapply(function(r, c) elev_matrix[r, c], row_pos, col_pos)
    render_path(
      extent    = ext,
      lat       = seg[, 2],
      long      = seg[, 1],
      altitude  = terrain_elev + 800,  # 800m above terrain surface
      zscale    = zscale,
      color     = "red",
      linewidth = 8,
      heightmap = elev_matrix
    )
  }
}

cat("France-Spain border drawn.\n")
cat("3D scene rendered.\n")

Reading ne_50m_admin_0_boundary_lines_land.zip from naturalearth...


France-Spain border drawn.
3D scene rendered.


### Step 7: Save the Final Image

We use `render_snapshot()` to capture the current rgl scene and save it as a PNG. This is an OpenGL snapshot — instant and reliable. If you want a higher-quality path-traced render instead, see the note in the appendix about `render_highquality()` and the additional dependencies it requires.

In [12]:
render_snapshot(
  filename = "aqi_pyrenees_3d.png",
  clear    = FALSE
)

cat("Render complete: aqi_pyrenees_3d.png\n")

Render complete: aqi_pyrenees_3d.png


### Step 8: Locate the Output

The PNG is saved to your working directory. Run `getwd()` to find the exact path, then open it from Finder.